In [0]:
from pyspark.sql.functions import col, lit

claim_schema = """
claim_id int,
policy_id int,
date_of_claim timestamp,
claim_amount long,
claim_status string,
LastUpdatedTimeStamp timestamp
"""

df = spark.read \
    .schema(claim_schema) \
    .parquet("abfss://landing@projpolicysytem.dfs.core.windows.net/ClaimData/*.parquet")

In [0]:
df = df.withColumn("claim_amount", col("claim_amount").cast("double"))

df_with_flag = df.withColumn("merge_flag", lit(False))

bronze_path = "abfss://bronzelayer@projpolicysytem.dfs.core.windows.net/Claim"

df_with_flag.write \
    .format("delta") \
    .mode("append") \
    .option("path", bronze_path) \
    .saveAsTable("policyprojcatalog.policyprojdb.Claim")

In [0]:
%sql
select * from policyprojcatalog.policyprojdb.Claim

In [0]:
dbutils.fs.mv(
    "abfss://processed@projpolicysytem.dfs.core.windows.net/ClaimData/",
    "abfss://landing@projpolicysytem.dfs.core.windows.net/ClaimData/",
    True
)

In [0]:
from datetime import datetime

current_time = datetime.now().strftime('%m-%d-%Y')

new_folder = f"abfss://processed@projpolicysytem.dfs.core.windows.net/ClaimData/{current_time}"

dbutils.fs.mv(
    "abfss://landing@projpolicysytem.dfs.core.windows.net/ClaimData/",
    new_folder,
    True
)